# Optimize a Data Center
# Оптимизация дата-центра

Мотивация: Data center design is a multi factor optimization problem. Surely, **we want to get as much computing
capacity as possible** — the services need a lot of resources today and will be asking for more tomorrow.
**But maximizing the raw capacity is not the only goal: it’s equally important to ensure that the computing
capacity is provided reliably even in the face of inevitable hardware failures.**

Что я прочитал и как в целом бы описал задачу - у нас есть дата-центр, основную работу там выполняют серверы. Серверы располагаются рядами, каждый отдельный сервер лежит в отдельном слоте (слоты могут быть разных размеров, так как каждый сервер определяется упорядоченной парой $<$размеры, вместимость$>$, где размеры - это физические параметры сервера, описанные количеством занимаемых им слотов, а вместимость - это сколько у нас внутри сервера процессоров).  

Что еще важно, помимо рядов у нас есть пулы серверов - некие логические объединения, и основную важность для нас представляют именно они. Потому как по идее пулы состоят из серверов, находящихся на разных рядах (так как если на одном ряде сервер выйдет из строя, то все сервера на ряде тоже станут недоступными).  

Теперь мы вводим понятие гарантированной вместимости (или вычислительности, как угодно) - это минимальное значение вместимости пула в той ситуации, когда не работет только один ряд серверов.  

Соответственно нам нужно максимизировать вот такое выражение $min(gc_1, gc_2, ... ,gc_n)$, где $gc_i$ - гарантированная **capacity** для пула $i$. Соответственно наш **score** - это и есть значение функции минимума.

## Формальная сторона вопроса.

### Множества и индексы
$R \in \{1, 2, ..., 1000\}$ - количество рядов в дата-центре  
$S \in \{1, 2, ..., 1000\}$ - количество слотов в каждом ряде  
$U \in \{0, ..., R \cdot S\}$ `(тут видимо декартово произведение имеется ввиду)`- количество недоступных слотов (на каждом ряду есть слоты, куда ничего положить нельзя)  
$P \in \{1, 2, ..., 1000\}$ - количество необходимых к созданию пулов  
$M \in \{1, ..., R \cdot S\}$ - количество серверов, которые необходимо разместить  

После "$U$" у нас идет $U$ упорядоченных пар $<r_i, s_i>$, которые указывают на то, что в ряду $r_i$ недоступен слот $s_i$  
  
После "$M$" у нас идет $M$ упорядоченных пар $<z_i, c_i>$, которые указывают на то, что сервер $i$ имеет размер $z_i$ и мощность $c_i$  

Прием как я понял какие-то сервера еще могут оказаться и не размещенными...

Ну и что-то вроде предлагаемой от автором целевой функции:  
$$gc_i = \min_{0 \leq r < R} (\sum_{k=0, \ server \ k \ in \ pool \ i}^{M-1}(c_i) \ \ \ - \sum_{k=0, \ server \ k \ in \ pool \ i, \ server \ k \ in \ row \ r}^{M-1}(c_i))$$

$$score \ = \ \min_{0 \leq i < P}(gc_i) \rightarrow max$$

Тогда  
$i \in \{0,...,M-1\}$ - серверы ($I$)  
$j \in \{0,...,P-1\}$ - пулы ($J$)  
$k \in \{0,...,R-1\}$ - ряды ($K$)  
$s \in \{0,...,S-1\}$ - слоты внутри ряда ($S_k$)  

$z_i$ - размер сервера $i$  
$c_i$ - мощность сервера $i$  

# Переменные  

$a_i \in \{0,1\}$ - сервер $i$ размещен или не размещен  
$p_{i,j} \in \{0,1\}$ - сервер $i$ закреплен за пулом $j$ или не закреплен  
$r_{i,k} \in \{0,1\}$ - сервер $i$ находится в ряду $k$ или не находится  
$x_{i,k,s} \in \{0,1\}$ - сервер $i$ начинается в ряду $k$ со слота $s$ или не начинается  
$y_{i,j,k} \in \{0,1\}$ - сервер $i$ одновременно в пуле j в ряду $k$ (линеаризация произведения $p_{i,j} \cdot r_{i, k}$)  
$C_{j, k} \leq 0$ - суммарная мощность пула $j$ в ряду $k$  
$M_{j} \leq 0$ - максимальная мощность пула $j$ в одном ряду  
$T \geq 0$ - минимальная гарантированная вместимость $\Longrightarrow$ Целевая функция: $T \rightarrow \max$  

# Ограничения  
Сервер либо размещен **только один раз**, либо не размещен вовсе  
$$\sum_{k=0}^{R-1}\sum_{s=0}^{S-1}x_{i,k,s} = a_i$$

Если сервер размещен, то он принадлежит только одному пулу  
$$\sum_{j=0}^{P-1}p_{i,j} = a_i, \ \forall i \in I$$

Если сервер размещен, то он находится только в одном ряду  

$$\sum_{k=0}^{R-1}r_{i,k} = a_i, \ \forall i \in I$$

Связь между $r_{i,k}$ и $x_{i,k,s}$  
$$r_{i,k} = \sum_{s=0}^{S-1}x_{i,k,s} \ \forall i \in I, \forall k \in K$$

Сервер не может находится за границами ряда:  
$$x_{i,k,s} = 0 \Longleftarrow s + z_i > S$$

Сервер не может занимать недоступные слоты  
$$x_{i,k,s} = 0 \Longleftarrow \exists s' \in [s, s + z_i - 1]: (k,s') \in U$$

Сервера не могут пересекаться  
$$\sum_{i=0}^{M-1}\sum_{s': s' \leq s < s' + z_i}x_{i,k,s'} \leq 1, \forall k \in K, \forall s \in S_k$$

Линеаризация произведения $p_{i,j} \cdot r_{i,k}$  
$$\begin{cases} y_{i,j,k} \leq p_{i,j} \\ y_{i,j,k} \leq r_{i,k} \\ y_{i,j,k} \geq p_{i,j} + r_{i,k} - 1 \end{cases} \forall i \in I, \forall j \in J, \forall k \in K$$

Мощность пула в ряду  
$$C_{j,k} = \sum_{i=0}^{M-1}c_i \cdot y_{i,j,k}, \forall j \in J, \forall k \in K$$

Максимум по ряду для пула  
$$M_{j} \geq C_{j,k}, \forall j \in J, \forall k \ in K$$

Гарантированная мощность пула  
$$T \leq \sum_{k=0}^{R-1}C_{j,k} - Mj, \forall j \in J$$

# Целевая функция  
$$T \rightarrow max$$

In [3]:
import os
import time
from ortools.linear_solver import pywraplp

INPUT_PATH = "labs/lab_3_1/input.txt"
OUTPUT_PATH = "labs/lab_3_1/output_scip.txt"
TIME_LIMIT_MS = 3_600_000 

In [4]:
def parse_file(filepath):
    with open(filepath) as f:
        data = f.read().strip().split()
    it = iter(data)
    R = int(next(it)); S = int(next(it)); U = int(next(it))
    P = int(next(it)); M = int(next(it))

    unavailable = set()
    for _ in range(U):
        r = int(next(it)); s = int(next(it))
        unavailable.add((r, s))

    servers = []
    for _ in range(M):
        z = int(next(it)); c = int(next(it))
        servers.append((z, c))

    return R, S, U, P, M, unavailable, servers

In [5]:
def write_output(path, allocation, M):
    with open(path, "w") as f:
        for i in range(M):
            if allocation[i] is None:
                f.write("x\n")
            else:
                r, s, p = allocation[i]
                f.write(f"{r} {s} {p}\n")

In [ ]:
def solve_file(filepath):
    R, S, U, P, M, unavailable, servers = parse_file(filepath)

    total_cap = sum(c for _, c in servers)

    solver = pywraplp.Solver.CreateSolver("SCIP")
    if not solver:
        return None, 0, "NO_SOLVER", R, S, P, M, 0

    solver.SetTimeLimit(TIME_LIMIT_MS)

    # переменные

    # a[i] — сервер i размещён
    a = {i: solver.BoolVar(f"a_{i}") for i in range(M)}

    # p[i,j] — сервер i в пуле j
    p = {}
    for i in range(M):
        for j in range(P):
            p[i, j] = solver.BoolVar(f"p_{i}_{j}")

    # r[i,k] — сервер i в ряду k
    r = {}
    for i in range(M):
        for k in range(R):
            r[i, k] = solver.BoolVar(f"r_{i}_{k}")

    # x[i,k,s] — сервер i начинается в ряду k со слота s
    x = {}
    for i in range(M):
        z_i = servers[i][0]
        for k in range(R):
            for s in range(S - z_i + 1):
                # проверяем, что сервер не накрывает недоступный слот
                ok = True
                for d in range(z_i):
                    if (k, s + d) in unavailable:
                        ok = False
                        break
                if ok:
                    x[i, k, s] = solver.BoolVar(f"x_{i}_{k}_{s}")

    # y[i,j,k] — сервер i и в пуле j, и в ряду k
    y = {}
    for i in range(M):
        for j in range(P):
            for k in range(R):
                y[i, j, k] = solver.BoolVar(f"y_{i}_{j}_{k}")

    # C[j,k] — capacity пула j в ряду k
    C = {}
    for j in range(P):
        for k in range(R):
            C[j, k] = solver.NumVar(0, total_cap, f"C_{j}_{k}")

    # Mj[j] — максимум capacity пула j по ряду
    Mj = {}
    for j in range(P):
        Mj[j] = solver.NumVar(0, total_cap, f"Mj_{j}")

    # T — целевая
    T = solver.NumVar(0, total_cap, "T")

    # ограничения

    # сервер либо размещён ровно один раз, либо нет
    for i in range(M):
        xs = [x[i, k, s] for (k, s) in [(kk, ss) for kk in range(R)
                                        for ss in range(S - servers[i][0] + 1)
                                        if (i, kk, ss) in x]]
        solver.Add(sum(xs) == a[i])

    # ровно один пул, если размещён
    for i in range(M):
        solver.Add(sum(p[i, j] for j in range(P)) == a[i])

    # ровно один ряд, если размещён
    for i in range(M):
        solver.Add(sum(r[i, k] for k in range(R)) == a[i])

    # связь r[i,k] и x[i,k,s]
    for i in range(M):
        for k in range(R):
            xs = [x[i, k, s] for s in range(S - servers[i][0] + 1)
                  if (i, k, s) in x]
            solver.Add(r[i, k] == sum(xs))

    # нет пересечений слотов в одном ряду
    for k in range(R):
        for s in range(S):
            terms = []
            for i in range(M):
                z_i = servers[i][0]
                for s0 in range(max(0, s - z_i + 1), min(s, S - z_i) + 1):
                    if (i, k, s0) in x:
                        terms.append(x[i, k, s0])
            if terms:
                solver.Add(sum(terms) <= 1)

    # линеаризация y = p AND r
    for i in range(M):
        for j in range(P):
            for k in range(R):
                solver.Add(y[i, j, k] <= p[i, j])
                solver.Add(y[i, j, k] <= r[i, k])
                solver.Add(y[i, j, k] >= p[i, j] + r[i, k] - 1)

    # C[j,k] = сумма capacity серверов, которые в пуле j и в ряду k
    for j in range(P):
        for k in range(R):
            terms = [servers[i][1] * y[i, j, k] for i in range(M)]
            solver.Add(C[j, k] == sum(terms))

    # Mj[j] >= C[j,k]
    for j in range(P):
        for k in range(R):
            solver.Add(Mj[j] >= C[j, k])

    # T <= sum_k C[j,k] - Mj[j] для каждого пула
    for j in range(P):
        solver.Add(sum(C[j, k] for k in range(R)) - Mj[j] >= T)

    # целевая функция
    objective = solver.Objective()
    objective.SetCoefficient(T, 1)
    objective.SetMaximization()

    print(f"[SCIP] vars={solver.NumVariables()}, "
          f"cons={solver.NumConstraints()}, "
          f"limit={TIME_LIMIT_MS / 1000:.0f}s")

    status = solver.Solve()

    if status == pywraplp.Solver.OPTIMAL:
        status_str = "OPTIMAL"
    elif status == pywraplp.Solver.FEASIBLE:
        status_str = "FEASIBLE"
    elif status == pywraplp.Solver.INFEASIBLE:
        status_str = "INFEASIBLE"
    else:
        status_str = f"FAILED({status})"

    if status not in (pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE):
        return None, 0, status_str, R, S, P, M, 0

    # решение
    allocation = [None] * M
    for i in range(M):
        if a[i].solution_value() < 0.5:
            continue
        # ряд
        row_i = None
        for k in range(R):
            if r[i, k].solution_value() > 0.5:
                row_i = k
                break
        # стартовый слот
        s_i = None
        z_i = servers[i][0]
        for k in range(R):
            for s in range(S - z_i + 1):
                if (i, k, s) in x and x[i, k, s].solution_value() > 0.5:
                    s_i = s
                    break
            if s_i is not None:
                break
        # пул
        pool_i = None
        for j in range(P):
            if p[i, j].solution_value() > 0.5:
                pool_i = j
                break
        if row_i is not None and s_i is not None and pool_i is not None:
            allocation[i] = (row_i, s_i, pool_i)

    placed = sum(1 for a_ in allocation if a_ is not None)
    score = round(T.solution_value())

    return allocation, score, status_str, R, S, P, M, placed


def main():
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    print(f"FILE: {INPUT_PATH}")

    t0 = time.time()
    alloc, score, status_str, R, S, P, M, placed = solve_file(INPUT_PATH)
    elapsed = time.time() - t0

    if alloc is None:
        print(f"R={R}, S={S}, P={P}, M={M}")
        print(f"статус: {status_str}")
        print("Решение не найдено.")
        return

    write_output(OUTPUT_PATH, alloc, M)

    print(f"R={R}, S={S}, P={P}, M={M}")
    print(f"status: {status_str}")
    print(f"score: {score}")
    print(f"placed: {placed}/{M}")
    print(f"время: {elapsed:.1f} сек")


if __name__ == "__main__":
    main()

FILE: labs/lab_3_1/input.txt
[SCIP] vars=1328043, cons=1364880, limit=3600s
R=16, S=100, P=45, M=625
status: FEASIBLE
score: 0
placed: 0/625
время: 3661.8 сек

Отсортируем сервера в порядке убывания мощности, будем выбирать самый слабый пул, в нем искать самый слабый ряд, в которой можно положить сервер, ну и если не получается, то ищем другие ряды в этом пуле, если не выходит, то смотрит другие пулы, если не выходит - то это какой-то странный сервер и размещать его мы не будем.

In [6]:
def function(R, S, P, M, unavailable, servers):
    # row_free[k][s] = True, если слот s в ряду k свободен
    row_free = [[True] * S for _ in range(R)]
    for r, s in unavailable:
        row_free[r][s] = False

    # суммарная capacity пула
    pool_total = [0] * P 

    # capacity пула в ряду                     
    pool_row_cap = [[0] * R for _ in range(P)]  

    # максимум по ряду (самый важный по capacity ряд в пуле, один сервер выйдет - и весь ряд выйдет)
    pool_max_row = [0] * P     

    # guaranteed capacity               
    pool_gc = [0] * P                         

    alloc = [None] * M

    # сортируем по убыванию capacity, при равенстве — по убыванию размера
    order = sorted(range(M), key=lambda i: (-servers[i][1], -servers[i][0]))

    for i in order:
        z, c = servers[i]

        # ищем пул с наименьшей gc
        best_pool = min(range(P), key=lambda j: pool_gc[j])

        # внутри пула — ищем ряд с наименьшей capacity этого пула
        # чтобы не концентрировать сервера в одном ряду
        row_order = sorted(range(R), key=lambda k: pool_row_cap[best_pool][k])

        placed = False
        for k in row_order:
            # ищем первое свободное окно длины z
            run = 0
            for s in range(S):
                if row_free[k][s]:
                    run += 1
                    if run == z:
                        start = s - z + 1
                        # нашли — кладём
                        for d in range(z):
                            row_free[k][start + d] = False

                        pool_total[best_pool] += c
                        pool_row_cap[best_pool][k] += c
                        pool_max_row[best_pool] = max(
                            pool_max_row[best_pool],
                            pool_row_cap[best_pool][k],
                        )
                        pool_gc[best_pool] = (
                            pool_total[best_pool] - pool_max_row[best_pool]
                        )

                        alloc[i] = (k, start, best_pool)
                        placed = True
                        break
                else:
                    run = 0
            if placed:
                break

        # если не влез — пробуем другие пулы
        if not placed:
            for pj in sorted(range(P), key=lambda j: pool_gc[j]):
                if pj == best_pool:
                    continue
                row_order = sorted(range(R),
                                   key=lambda k: pool_row_cap[pj][k])
                for k in row_order:
                    run = 0
                    for s in range(S):
                        if row_free[k][s]:
                            run += 1
                            if run == z:
                                start = s - z + 1
                                for d in range(z):
                                    row_free[k][start + d] = False

                                pool_total[pj] += c
                                pool_row_cap[pj][k] += c
                                pool_max_row[pj] = max(
                                    pool_max_row[pj],
                                    pool_row_cap[pj][k],
                                )
                                pool_gc[pj] = (
                                    pool_total[pj] - pool_max_row[pj]
                                )

                                alloc[i] = (k, start, pj)
                                placed = True
                                break
                        else:
                            run = 0
                    if placed:
                        break
                if placed:
                    break

    return alloc, min(pool_gc), pool_gc

In [7]:
def main():
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    R, S, U, P, M, unavailable, servers = parse_file(INPUT_PATH)
    print(f"R={R}, S={S}, U={U}, P={P}, M={M}")

    t0 = time.time()
    alloc, score, pool_gc = function(R, S, P, M, unavailable, servers)
    elapsed = time.time() - t0

    write_output(OUTPUT_PATH, alloc, M)

    placed = sum(1 for a in alloc if a is not None)
    print(f"score: {score}")
    print(f"placed: {placed}/{M}")
    print(f"время: {elapsed:.2f} сек")
    print(f"gc по пулам: {sorted(pool_gc)}")


if __name__ == "__main__":
    main()

R=16, S=100, U=80, P=45, M=625
score: 377
placed: 433/625
время: 0.67 сек
gc по пулам: [377, 377, 377, 377, 377, 377, 377, 377, 378, 378, 378, 378, 378, 378, 378, 378, 378, 392, 392, 393, 394, 394, 394, 394, 394, 394, 394, 394, 394, 394, 395, 395, 395, 395, 395, 395, 395, 395, 395, 395, 395, 395, 395, 395, 395]
